In [11]:
import polars as pl

catalog_df = pl.read_parquet("data/embedded_drug_catalog.parquet")

# Préfixes ATC strictement locaux, inertes ou physiologiques
SAFE_ATC_PREFIXES = (
    "D",       # Dermatologie complète (crèmes, pommades, émollients, topiques)
    "S",       # Organes sensoriels (collyres ophtalmiques, gouttes auriculaires)
    "A02A",    # Antiacides locaux (sels d'aluminium, magnésium, carbonate de calcium)
    "A06A",    # Laxatifs (osmotiques, de lest, suppositoires glycérine, PEG)
    "B05",     # Substituts du sang, solutés d'irrigation, électrolytes IV (NaCl, Ringer, KCl)
    "A11",     # Vitamines (acide ascorbique, vitamine D, B12, etc.)
    "A12",     # Suppléments minéraux (calcium, potassium, magnésium oraux)
    "V04",     # Agents de diagnostic
    "V07",     # Produits techniques / solvants médicaux
    "J07",     # Vaccins (détectés si administrés avec recul temporel)
)

# Mots-clés de formulations ou substances inertes / physiologiques dans RxNorm
SAFE_KEYWORDS = [
    "acetaminophen", "paracetamol",
    "sodium chloride", "sterile water", "dextrose",
    "potassium chloride", "calcium carbonate", "magnesium hydroxide",
    "polyethylene glycol", "glycerin", "lactulose", "senna",
    "lidocaine", "chlorhexidine", "povidone-iodine",
    "artificial tears", "petrolatum", "zinc oxide", "hydrocortisone topical"
]

class_k_enriched = []
for row in catalog_df.iter_rows(named=True):
    cid = row["rxnorm_concept_id"]
    raw_atc = row.get("atc_codes")
    atc_list = raw_atc.to_list() if hasattr(raw_atc, "to_list") else (raw_atc or [])
    name = str(row.get("rxnorm_name", "")).lower()

    # 1. Match par préfixe ATC inerte/topique
    matches_atc = any(
        any(str(a).startswith(pfx) for pfx in SAFE_ATC_PREFIXES)
        for a in atc_list if a
    )

    # 2. Match par substance physiologique / inerte
    matches_name = any(kw in name for kw in SAFE_KEYWORDS)

    # Sécurité absolue : exclure tout corticoïde systémique oral/IV ou immunomodulateur
    is_systemic_immune = any(
        str(a).startswith("H02AB") or str(a).startswith("L") 
        for a in atc_list if a
    )

    if (matches_atc or matches_name) and not is_systemic_immune:
        class_k_enriched.append(cid)

print(f"Molécules dans la Classe K enrichie : {len(class_k_enriched)} / {catalog_df.height} ({len(class_k_enriched)/catalog_df.height*100:.1f}%)")

Molécules dans la Classe K enrichie : 452 / 2457 (18.4%)
